# Person 2: Recurrent model (BiGRU / BiLSTM over MFCC frame sequences)

This notebook runs `scripts/run_person2_recurrent.py`. All model code lives in `src/models/recurrent.py` and `src/preprocessing/sequences.py`. The notebook does not reimplement loading, the split, or preprocessing.

**Rules followed:** only `data/splits/shared_split.csv` is used (70/15/15, seed 42). Every tuning and early-stopping decision uses the validation split. The internal test split (`split == "test"`) is evaluated once, after the configuration is locked. The official Zindi `Test.csv` is never used.

**Runtime:** use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). A full run takes roughly 1–2 hours on a T4. On a laptop CPU it would take days.

**Before running in Colab:** upload `Swahili_words.zip`, `Train.csv`, `Test.csv`, and `SampleSubmission.csv` to one Google Drive folder, and set `DRIVE_DATA_DIR` below to that folder.

In [ ]:
# --- Settings ---------------------------------------------------------------
REPO_URL = "https://github.com/Orrie-Dan/NLP-and-Language-Technologies.git"
BRANCH = "eddy"                                   # branch that contains the Person 2 code
DRIVE_DATA_DIR = "/content/drive/MyDrive/swahili_audio"   # folder holding the 4 data files
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/swahili_audio/person2_results"

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/NLP-and-Language-Technologies")
    if not repo.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    raw = repo / "data" / "raw"
    for name in ["Swahili_words.zip", "Train.csv", "Test.csv", "SampleSubmission.csv"]:
        if not (raw / name).exists():
            shutil.copy(Path(DRIVE_DATA_DIR) / name, raw / name)
else:
    # Running locally from notebooks/: move to the repository root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")
print("Working directory:", Path.cwd())
print(sorted(p.name for p in Path("data/raw").iterdir()))

In [ ]:
%pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## Run the experiment

The first time, this extracts 13-coefficient MFCC frame sequences for all 4,200 labelled clips (cached in `results/models/recurrent/cache/`). It then:

1. **Validation search.** One factor changes at a time and the best validation macro-F1 is carried forward: cell and direction (BiGRU, BiLSTM, GRU, LSTM), input features (static MFCC vs MFCC + Δ + ΔΔ), temporal resolution (10 ms frames vs 3 stacked frames = 30 ms), pooling (masked mean vs last state), hidden size (64/128/256), and dropout (0.15/0.3/0.5).
2. **Locked test evaluation.** The selected model is evaluated once on the internal test split.
3. **Seed robustness.** The same locked configuration is retrained with seeds 43 and 44.

In [ ]:
!python scripts/run_person2_recurrent.py --device auto

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

out = Path("results/models/recurrent")
fig = Path("results/figures/recurrent")
pd.set_option("display.precision", 4)

print("Validation search (all runs):")
display(pd.read_csv(out / "validation_experiments.csv")[
    ["stage", "run", "same_as", "cell", "bidirectional", "feature_streams", "frame_stack",
     "pooling", "hidden_size", "dropout", "best_epoch", "n_parameters", "val_accuracy", "val_macro_f1"]])

print("Selected configuration:")
print(json.dumps(json.loads((out / "selected_config.json").read_text()), indent=2))

In [ ]:
final = json.loads((out / "final_recurrent_results.json").read_text())
classical = pd.read_csv("results/classical/final_classical_results.csv")
comparison = pd.concat([
    classical[["model", "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]],
    pd.DataFrame([{"model": f"recurrent ({final['selected_run']})", **final["test"]}]),
], ignore_index=True)
print("Internal test comparison:")
display(comparison)
print("Seed robustness (locked configuration):")
display(pd.read_csv(out / "seed_robustness.csv"))
print("Accuracy by clip duration (internal test):")
display(pd.read_csv(out / "test_accuracy_by_duration.csv"))

In [ ]:
for name in ["validation_experiments", "learning_curves", "test_confusion_matrix", "test_roc_curves",
             "test_per_class_f1_vs_classical", "test_accuracy_by_duration"]:
    print(name)
    display(Image(filename=str(fig / f"{name}.png"), width=720))

## Save results to Drive (Colab only)

The Colab disk is wiped when the session ends. This copies the results and figures to Drive, without the feature cache or checkpoints. Download the folder, then copy `results/models/recurrent/` and `results/figures/recurrent/` into the repository and commit them.

In [ ]:
if IN_COLAB:
    dest = Path(DRIVE_RESULTS_DIR)
    for src in [out, fig]:
        target = dest / src
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(src, target, ignore=shutil.ignore_patterns("cache", "checkpoints"))
    print("Saved to", dest)